# Day 19 — Frontend, UX & the AI Venture Intelligence Dashboard

A React + TypeScript SPA (Founder & Investor portals) over the FastAPI backend — a **thin, typed
presenter**: zero ML in the client, it renders the backend's typed JSON, and the honesty boundary
(`honesty_note`, `deferred`, per-section `source`) is shown as UI, never faked.

**Honest note:** a Python notebook can't render React. So this notebook (1) **verifies the frontend
builds + typechecks** (`npm run build`), (2) documents the component structure, and (3) shows the **live
API responses** each component consumes (via TestClient). To actually use the UI: `npm run dev` +
`CORS_ORIGINS=http://localhost:5173 uvicorn src.api.main:app`.

In [1]:
import os, sys, subprocess, shutil, json
os.environ["OMP_NUM_THREADS"] = "1"          # before any faiss/torch import (TestClient loads both)
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.abspath(".."))
sys.path.insert(0, os.getcwd())
print("repo:", os.getcwd())

repo: /Users/Yash/AI-Ventutre-Intelligence-Platform


## 1 — The frontend, at a glance

The components built in Phases A–K (each a typed presenter of one backend endpoint).

In [2]:
from pathlib import Path
src = Path("frontend/src")
for p in sorted(src.rglob("*.ts*")):
    if p.name.endswith(".d.ts"): continue
    print(" ", p.relative_to("frontend"))

  src/App.tsx
  src/api/client.ts
  src/api/types.ts
  src/components/CampaignForm.tsx
  src/components/Dashboard/DeferredScores.tsx
  src/components/Dashboard/PredictionPanel.tsx
  src/components/Dashboard/RecommendationPanel.tsx
  src/components/Explainability/ExplanationPanel.tsx
  src/components/FounderAnalysis.tsx
  src/components/Investor/InvestorAnalysis.tsx
  src/components/Layout/Header.tsx
  src/components/ProtectedRoute.tsx
  src/components/Reports/ReportViewer.tsx
  src/components/Retrieval/RetrievalDashboard.tsx
  src/components/ScenarioSimulator/ScenarioSimulator.tsx
  src/components/ui.tsx
  src/hooks/useApi.ts
  src/hooks/useAuth.tsx
  src/hooks/useDebounced.ts
  src/lib/categories.ts
  src/main.tsx
  src/pages/FounderPortal.tsx
  src/pages/Home.tsx
  src/pages/InvestorPortal.tsx
  src/pages/Login.tsx


## 2 — It builds + typechecks (`npm run build`)

The build runs `tsc -b && vite build`, so a passing build means the whole app **typechecks** against the
API contract — a field the backend doesn't return would be a compile error.

In [3]:
npm = shutil.which("npm")
if not npm:
    print("npm not found on PATH — run `npm run build` in frontend/ manually")
else:
    out = subprocess.run([npm, "run", "build"], cwd="frontend", capture_output=True, text=True)
    tail = [l for l in out.stdout.splitlines() if l.strip()][-8:]
    print("\n".join(tail))
    print("\nbuild exit code:", out.returncode, "->", "GREEN ✅" if out.returncode == 0 else "FAILED")

transforming...✓ 672 modules transformed.
rendering chunks...
computing gzip size...
dist/index.html                             0.45 kB │ gzip:   0.29 kB
dist/assets/index-CZIvDmrD.css             22.51 kB │ gzip:   5.15 kB
dist/assets/index-BR9L3vI5.js             308.00 kB │ gzip:  99.06 kB
dist/assets/ExplanationPanel-CG929ELd.js  354.69 kB │ gzip: 103.01 kB
✓ built in 190ms

build exit code: 0 -> GREEN ✅


## 3 — The API contract each component renders (live)

Every panel is a typed view of one endpoint. Here are the real responses the UI consumes.
*(Boots the full warm stack — ~tens of seconds.)*

In [4]:
from fastapi.testclient import TestClient
from src.api.main import app
C = {"name": "Artificer", "main_category": "Games", "usd_goal_real": 30000.0,
     "launched": "2016-05-01 12:00:00", "deadline": "2016-05-31 12:00:00"}

with TestClient(app) as c:
    pred = c.post("/predict", json=C).json()
    print("PredictionPanel  ← /predict   :", {k: pred[k] for k in ("funding_probability", "predicted_label", "decision_threshold")})
    ex = c.post("/explain", json=C).json()
    print("ExplanationPanel ← /explain   : top_positive[0] =", ex["top_positive"][0]["descriptor"], "| honesty:", (ex["honesty_note"] or "")[:40], "…")
    rt = c.post("/retrieve?k=5", json=C).json()
    print("RetrievalDash    ← /retrieve  :", len(rt["similar_successful"]), "funded /", len(rt["similar_failed"]), "failed similar")
    rc = c.post("/recommend", json=C).json()
    print("RecommendPanel   ← /recommend : readiness", rc["investment_readiness"]["stage"], "| improvements:", len(rc["priority_improvements"]))
    an = c.post("/analyze", json=C).json()
    print("InvestorAnalysis ← /analyze   :", an["committee_recommendation"], "| confidence", round(an["overall_confidence"], 3))
    rp = c.post("/report/founder", json=C).json()
    print("ReportViewer     ← /report    :", len(rp["sections"]), "sections; deferred_appendix:", rp["deferred_appendix"]["note"][:50], "…")

/Users/Yash/AI-Ventutre-Intelligence-Platform/.venv/lib/python3.14/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


2026-08-17 11:57:14 | avip.api                         | INFO     | API starting — OMP_NUM_THREADS=1, log_file=None, level=INFO


2026-08-17 11:57:16 | avip.core.registry               | INFO     | PredictionEngine ready — model=calibrated_model.joblib, threshold=0.329.


2026-08-17 11:57:17 | avip.core.registry               | INFO     | Item frame: 265171 campaigns, 13 item features, base funding rate 0.4038.


2026-08-17 11:57:17 | avip.core.registry               | INFO     | CounterfactualPipeline ready (warm model + item frame).


2026-08-17 11:57:17 | avip.core.registry               | INFO     | Loaded LGBMClassifier (865 features).


... (168 outputs left out)

2026-08-17 11:57:50 | avip.core.registry               | INFO     | VentureDecision: Artificer | Monitor Progress | readiness 27 | confidence 0.83.


2026-08-17 11:57:50 | avip.core.registry               | INFO     | Founder report: Artificer (8 sections, readiness 27).


2026-08-17 11:57:50 | avip.api                         | INFO     | [8980c944] POST /report/founder -> 200 (2561ms)


ReportViewer     ← /report    : 8 sections; deferred_appendix: Not covered — the dataset has no data for these; n …
2026-08-17 11:57:50 | avip.api                         | INFO     | API shutting down.


## 4 — Honesty is a UI element

The submission form offers ONLY the backend's real fields; fabricated inputs are rejected; deferred
scores and honesty notes are shown, not hidden.

In [5]:
from fastapi.testclient import TestClient
from src.api.main import app
with TestClient(app) as c:
    # the form's fields ARE the request contract; a fabricated field is 422-rejected
    ok = c.post("/predict", json=C).status_code
    bad = c.post("/predict", json={**C, "founder_experience": "expert", "mrr": 20000}).status_code
    print("real fields ->", ok, "| fabricated 'founder_experience'/'mrr' ->", bad, "(422 = rejected at the boundary)")
    # every response carries a rendered honesty boundary
    r = c.post("/recommend", json=C).json()
    print("deferred keys the UI greys out:", list(r["deferred"])[:6])

2026-08-17 11:57:50 | avip.api                         | INFO     | API starting — OMP_NUM_THREADS=1, log_file=None, level=INFO


2026-08-17 11:57:50 | avip.api                         | INFO     | API ready — registry loaded.


2026-08-17 11:57:50 | avip.preprocessing.cleaning      | INFO     | TextCleaner: wrote cleaned titles to 'name_clean' (raw 'name' preserved).


2026-08-17 11:57:50 | avip.api                         | INFO     | [ece5354b] POST /predict -> 200 (16ms)


2026-08-17 11:57:50 | avip.api                         | INFO     | [1d5743bf] POST /predict -> 422 (2ms)


real fields -> 200 | fabricated 'founder_experience'/'mrr' -> 422 (422 = rejected at the boundary)
2026-08-17 11:57:50 | avip.preprocessing.cleaning      | INFO     | TextCleaner: wrote cleaned titles to 'name_clean' (raw 'name' preserved).


... (21 outputs left out)

2026-08-17 11:57:51 | avip.core.registry               | INFO     | Investor strategy: Monitor Progress (Fixable, 15% -> 39%).


2026-08-17 11:57:51 | avip.core.registry               | INFO     | VentureDecision: Artificer | Monitor Progress | readiness 27 | confidence 0.83.


2026-08-17 11:57:51 | avip.api                         | INFO     | [8078aece] POST /recommend -> 200 (1260ms)


deferred keys the UI greys out: ['customer_retention', 'monthly_revenue', 'revenue_growth', 'unit_economics', 'competitive_moat', 'named_competitors']
2026-08-17 11:57:51 | avip.api                         | INFO     | API shutting down.


## Summary — the pattern, and the honesty

**A thin typed presenter.** The frontend holds zero ML — it submits the real Kickstarter fields, renders
the backend's typed JSON, and shows the honesty boundary as UI: real scores as cards and deferred ones as
greyed "no data basis" placeholders, real scenario levers (not MRR/retention sliders), per-report-section
provenance, and neighbor-computed benchmarks. It builds and typechecks against the live contract, so a
field the API doesn't return can't be referenced. Auth is auth-ready (a stub login) and the catalog /
pattern panel / server-PDF are deferred — named, not faked. React runs in the browser via `npm run dev`;
this notebook verified the build and the contract, honest that it can't render the UI itself.

> **Interview soundbite.** *"Day 19 is a React-TypeScript frontend that's a thin, typed presenter over
> the FastAPI backend — zero ML in the client, and the types mirror the API schemas so a field the
> backend doesn't return is a compile error. The whole day's discipline is that honesty is a UI element:
> the form offers only the real Kickstarter fields because the backend 422-rejects anything else, the
> scores the model can't support render as greyed 'deferred' cards instead of fabricated gauges, and the
> scenario simulator's levers are real controllable features, not revenue and retention sliders that
> don't exist. A Python notebook can't render React, so I verified what it can — that the frontend builds
> and typechecks, and that each component's endpoint returns the shape it renders — and I said plainly
> that running the UI is npm run dev with the backend, rather than faking a screenshot."*